# 01 - XML Exploration & Model Testing

Exploratory notebook for:
1. Understanding LiverTox XML structure
2. Testing dataclass models
3. Prototyping parser and deterministic extraction logic
4. Testing LLM extraction prompts

In [4]:
import xml.etree.ElementTree as ET
from pathlib import Path
import re
import json

from livertox_extraction.models import (
    DrugExtraction, ParsedSections, validate_extraction, extraction_from_dict,
    VALID_DILI_SCORES, VALID_INJURY_PATTERNS, VALID_REGULATORY_STATUSES
)

XML_DIR = Path("/Users/maggiebrown/Desktop/axiom_bio/livertox-parser/data")
xml_files = sorted(XML_DIR.glob("*.xml"))
print(f"Found {len(xml_files)} XML files")
print([f.stem for f in xml_files[:10]], "...")

Found 85 XML files
['Acetaminophen', 'Ambrisentan', 'Amiodarone', 'Amitriptyline', 'Amodiaquine', 'Amoxicillin', 'Atorvastatin', 'Azathioprine', 'Benzbromarone', 'Benztropine'] ...


## 1. Parse a single XML and explore structure

In [5]:
# Pick a drug with rich content
sample_file = XML_DIR / "Zileuton.xml"
tree = ET.parse(sample_file)
root = tree.getroot()

# List all section IDs and titles
print("=== All sections ===")
for sec in root.iter("sec"):
    sid = sec.get("id", "(no id)")
    title_el = sec.find("title")
    title = title_el.text if title_el is not None else "(no title)"
    print(f"  {sid}: {title}")

=== All sections ===
  Zileuton.OVERVIEW: OVERVIEW
  Zileuton.Introduction: Introduction
  Zileuton.Background: Background
  Zileuton.Hepatotoxicity: Hepatotoxicity
  Zileuton.Mechanism_of_Liver_Injury: Mechanism of Liver Injury
  Zileuton.Outcome_and_Management: Outcome and Management
  Zileuton.CASE_REPORT: CASE REPORT
  Zileuton.Case_1_28_year_old_woman_with_s: Case 1. 28 year old woman with severe symptomatic liver injury during zileuton therapy.(
  Zileuton.Key_Points: Key Points
  Zileuton.Laboratory_Values: Laboratory Values
  Zileuton.Comment: Comment
  Zileuton.PRODUCT_INFORMATION: PRODUCT INFORMATION
  Zileuton.CHEMICAL_FORMULA_AND_STRUCTURE: CHEMICAL FORMULA AND STRUCTURE
  Zileuton.CITED_REFERENCE: CITED REFERENCE
  Zileuton.ANNOTATED_BIBLIOGRAPHY: ANNOTATED BIBLIOGRAPHY
  Zileuton.OTHER_REFERENCE_LINKS: OTHER REFERENCE LINKS


In [6]:
def get_section_text(root, section_id_contains):
    """Extract plain text from a section matching the ID pattern."""
    for sec in root.iter("sec"):
        sid = sec.get("id", "")
        if section_id_contains in sid:
            # Get all text content, stripping tags
            text = ET.tostring(sec, encoding="unicode", method="text")
            # Clean up whitespace
            text = re.sub(r"\s+", " ", text).strip()
            return text
    return None

# Test on key sections
for section in ["Hepatotoxicity", "Background", "Mechanism", "Outcome"]:
    text = get_section_text(root, section)
    if text:
        print(f"\n=== {section} (first 200 chars) ===")
        print(text[:200])
    else:
        print(f"\n=== {section}: NOT FOUND ===")


=== Hepatotoxicity (first 200 chars) ===
Hepatotoxicity In premarketing studies, zileuton therapy was found to be associated with mild-to-moderate serum aminotransferase elevations. In large prospective studies, ALT elevations above 3 times 

=== Background (first 200 chars) ===
Background Zileuton (zye loo' ton) is an inhibitor of the enzyme 5-lipoxygenase which is responsible for the conversion of arachidonic acid into leukotriene A4, a potent mediator of the inflammatory c

=== Mechanism (first 200 chars) ===
Mechanism of Liver Injury The mechanism of liver injury due to zileuton is unclear, however in vitro studies suggest that reactive metabolites of zileuton can covalently bind to intracellular proteins

=== Outcome (first 200 chars) ===
Outcome and Management The serum elevations in serum aminotransferase levels which occur during zileuton therapy usually resolve rapidly (1 to 4 weeks) once therapy is stopped. Cases of clinical appar


## 2. Test deterministic extraction patterns

In [7]:
hepatotox_text = get_section_text(root, "Hepatotoxicity")
print("Full hepatotoxicity section:")
print(hepatotox_text)

Full hepatotoxicity section:
Hepatotoxicity In premarketing studies, zileuton therapy was found to be associated with mild-to-moderate serum aminotransferase elevations. In large prospective studies, ALT elevations above 3 times the upper limit of the normal range occurred in 1.9% of patients treated with zileuton for at least one year, compared to 0.2% of placebo recipients. These elevations were usually transient, asymptomatic and rapidly reversible. However, some patients with ALT elevations reported symptoms suggestive of hepatic injury (fatigue, nausea, abdominal pain) and individual cases of clinically apparent, frank liver injury with jaundice were seen. The typical onset of liver enzyme elevations was within 4 to 8 weeks of starting zileuton, but cases arising after 6 months were also reported. In cases with jaundice, the pattern of serum enzyme elevations was hepatocellular. Immunoallergic and autoimmune features were not prominent. Recovery was rapid, usually within 1 to 2 mo

In [8]:
# DILI score regex
dili_pattern = r"Likelihood score:\s*([A-E]\*?|X)"
match = re.search(dili_pattern, hepatotox_text)
if match:
    score = match.group(1)
    print(f"DILI score: {score}")
    print(f"Valid? {score in VALID_DILI_SCORES}")
else:
    print("No DILI score found")

DILI score: D
Valid? True


In [9]:
# Enzyme elevation fraction regex
# Patterns: 'X% of patients', 'in X% of', 'occurred in X%'
enzyme_pattern = r"(\d+\.?\d*)%\s*of\s*(?:patients|recipients)"
matches = re.findall(enzyme_pattern, hepatotox_text, re.IGNORECASE)
print(f"Percentage mentions: {matches}")

# More contextual: look for percentages near ALT/aminotransferase/elevation keywords
context_pattern = r"(?:ALT|aminotransferase|enzyme).*?(\d+\.?\d*)%\s*of\s*(?:patients|recipients)"
match = re.search(context_pattern, hepatotox_text, re.IGNORECASE)
if match:
    pct = float(match.group(1))
    print(f"Enzyme elevation rate: {pct}% = {pct/100:.4f} as fraction")

Percentage mentions: ['1.9']
Enzyme elevation rate: 1.9% = 0.0190 as fraction


In [11]:
# Test DILI score extraction across all available XMLs
dili_pattern = r"Likelihood score:\s*([A-E]\*?|X)"

print("=== DILI scores across all drugs ===")
for xml_file in xml_files:
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        hepatotox = get_section_text(root, "Hepatotoxicity")
        if hepatotox:
            match = re.search(dili_pattern, hepatotox)
            score = match.group(1) if match else "NOT FOUND"
        else:
            score = "NO HEPATOTOX SECTION"
    except ET.ParseError as e:
        score = f"XML PARSE ERROR: {e}"
    print(f"  {xml_file.stem}: {score}")

=== DILI scores across all drugs ===
  Acetaminophen: A
  Ambrisentan: E
  Amiodarone: A
  Amitriptyline: B
  Amodiaquine: A
  Amoxicillin: B
  Atorvastatin: A
  Azathioprine: A
  Benzbromarone: B
  Benztropine: E
  Bosentan: C
  Buspirone: E
  Carbamazepine: A
  Cardizepam: NO HEPATOTOX SECTION
  Celecoxib: B
  Chlorpromazine: A
  Clomipramine: D
  Clozapine: A
  Cyclophosphamide: B
  Dabigatran: D
  Dermacillin: NO HEPATOTOX SECTION
  Desipramine: D
  Diclofenac: A
  Endocrinex: NO HEPATOTOX SECTION
  Entacapone: D
  Felbamate: B
  Felodipine: E
  Felodipinee: E
  Fenofibrate: B
  Flavoxate: NOT FOUND
  Fluoxetine: C
  Flutamide: A
  Gastrozine: NO HEPATOTOX SECTION
  Hematolix: NO HEPATOTOX SECTION
  HepC: XML PARSE ERROR: mismatched tag: line 68, column 4
  Hepatrofin: NO HEPATOTOX SECTION
  Ibuprofen: A
  Imipramine: B
  Immunexor: NO HEPATOTOX SECTION
  Indomethacin: C
  Itraconazole: B
  Ketoconazole: A
  Lapatinib: B
  Levofloxacin: A
  Lisinopril: B
  Meloxicam: C
  Metformin:

## 3. Explore case report Key Points tables

In [12]:
# Parse case report Key Points tables
sample_file = XML_DIR / "Zileuton.xml"
tree = ET.parse(sample_file)
root = tree.getroot()

for table_wrap in root.iter("table-wrap"):
    table_id = table_wrap.get("id", "")
    print(f"\nTable: {table_id}")
    for tr in table_wrap.iter("tr"):
        th = tr.find("th")
        td = tr.find("td")
        if th is not None and td is not None:
            key = (th.text or "").strip().rstrip(":")
            val_text = ET.tostring(td, encoding="unicode", method="text").strip()
            if key and val_text:
                print(f"  {key}: {val_text}")


Table: Zileuton.Tc
  Medication: Zileuton (400 mg four times daily)
  Pattern: Hepatocellular (R=25)
  Severity: 2+ (jaundice without hospitalization)
  Latency: 6 weeks (jaundice, increased enzymes)
  Recovery: 35-70 days

Table: Zileuton.Td


## 4. Test dataclass models

In [13]:
# Create a DrugExtraction from what we've found so far
zileuton = DrugExtraction(
    drug_name="Zileuton",
    dili_likelihood_score="D",
    injury_pattern="hepatocellular",
    fraction_patients_with_enzyme_elevation=0.019,
    peak_alt=33.0,
    r_ratio=25.0,
    is_immune_mediated=False,
    onset_time={"min": 4, "max": 8, "unit": "weeks"},
)

# Validate it
errors = validate_extraction(zileuton)
print(f"Validation errors: {errors}")

# Print as JSON
print("\nAs JSON:")
print(zileuton.to_json())

Validation errors: []

As JSON:
{
  "drug_name": "Zileuton",
  "dili_likelihood_score": "D",
  "injury_pattern": "hepatocellular",
  "fraction_patients_with_enzyme_elevation": 0.019,
  "fraction_patients_with_dili": null,
  "is_immune_mediated": false,
  "risk_factors": null,
  "safe_dose": null,
  "toxic_dose": null,
  "onset_time": {
    "min": 4,
    "max": 8,
    "unit": "weeks"
  },
  "peak_alt": 33.0,
  "peak_alp": null,
  "r_ratio": 25.0,
  "bilirubin_peak": null,
  "regulatory_status": null
}


In [14]:
# Test creating from a dict (simulates LLM output parsing)
llm_output = {
    "drug_name": "Itraconazole",
    "dili_likelihood_score": "B",
    "injury_pattern": "cholestatic",
    "fraction_patients_with_enzyme_elevation": 0.05,
    "is_immune_mediated": True,
    "risk_factors": [
        {"factor": "Pre-existing liver disease", "supporting_quote": "patients with active liver disease"}
    ],
    "some_extra_key": "this should be ignored",
}

drug = extraction_from_dict(llm_output)
print(f"Drug: {drug.drug_name}, DILI={drug.dili_likelihood_score}")
print(f"Risk factors: {drug.risk_factors}")
print(f"Validation: {validate_extraction(drug)}")

Drug: Itraconazole, DILI=B
Risk factors: [{'factor': 'Pre-existing liver disease', 'supporting_quote': 'patients with active liver disease'}]
Validation: []


In [15]:
# Test validation catches bad values
bad_drug = DrugExtraction(
    drug_name="BadDrug",
    dili_likelihood_score="F",              # invalid score
    fraction_patients_with_enzyme_elevation=1.5,  # out of range
    peak_alt=-10.0,                          # negative
)
errors = validate_extraction(bad_drug)
print("Errors caught:")
for e in errors:
    print(f"  - {e}")

Errors caught:
  - dili_likelihood_score 'F' not in ['A', 'B', 'C', 'D', 'E', 'E*', 'X']
  - fraction_patients_with_enzyme_elevation must be between 0.0 and 1.0, got 1.5
  - peak_alt must be non-negative, got -10.0


## 5. Test parser functions (after parser.py is built)

```python
from livertox_extraction.parser import parse_xml

sections = parse_xml(XML_DIR / "Zileuton.xml")
print(sections)
```

## 6. Test deterministic extraction (after deterministic.py is built)

```python
from livertox_extraction.deterministic import extract_deterministic

result = extract_deterministic(sections)
print(result)
```

## 7. Test LLM extraction (after llm_extractor.py is built)

```python
from livertox_extraction.llm_extractor import extract_with_llm

result = extract_with_llm(sections)
print(result)
```